In [7]:
import numpy as np 
import dgl 
from dgl.nn import GraphConv  
import torch
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
import dgl.data

FileNotFoundError: Cannot find DGL C++ graphbolt library at /opt/conda/envs/custom/lib/python3.8/site-packages/dgl/graphbolt/libgraphbolt_pytorch_1.13.0.so

In [4]:
import dgl

ModuleNotFoundError: No module named 'pydantic'

In [5]:
!pip install pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 5.8 MB/s eta 0:00:0000:0100:01m


1.Graphs


In [2]:
# edges 0->1, 0->2, 0->3, 1->3
u, v = torch.tensor([0,0,0,1]), torch.tensor([1,2,3,3])
g = dgl.graph((u, v))

In [3]:
print(g) # number of nodes are inferred from the max node IDs in the given edges

Graph(num_nodes=4, num_edges=4,
      ndata_schemes={}
      edata_schemes={})


In [4]:
g.nodes()

tensor([0, 1, 2, 3])

In [5]:
g.edges()

(tensor([0, 0, 0, 1]), tensor([1, 2, 3, 3]))

In [6]:
#edge end nodes and edge IDs
g.edges(form='all')

(tensor([0, 0, 0, 1]), tensor([1, 2, 3, 3]), tensor([0, 1, 2, 3]))

In [7]:
#if the node with the largest ID is isolated(meaning noedges), then one need to explicitly set the  number of nodes
g = dgl.graph((u,v), num_nodes=8)

In [ ]:
#for an undirected graph, one needs to create edges for both directions

In [8]:
bg = dgl.to_bidirected(g)

In [26]:
bg.edges()

(tensor([0, 0, 0, 1, 1, 2, 3, 3]), tensor([1, 2, 3, 0, 3, 0, 0, 1]))

In [ ]:
#node and edge feature

In [28]:
g = dgl.graph(([0, 0, 1, 5], [1, 2, 2, 0])) # 6 nodes, 4 edges

In [29]:
g

Graph(num_nodes=6, num_edges=4,
      ndata_schemes={}
      edata_schemes={})

In [31]:
g.ndata['x'] = torch.ones(g.num_nodes(), 3)  # node feature of length 3
g.edata['x'] = torch.ones(g.num_edges(), dtype=torch.int32) # scalar integer feature

In [1]:
from skt.vault_utils import get_secrets
proxies = get_secrets('proxies')
import os 
import numpy as np
os.environ['http_proxy'] = proxies['http']
os.environ['https_proxy'] = proxies['https']

In [4]:
dataset = dgl.data.CoraGraphDataset()

Extracting file to /home/x1113496/.dgl/cora_v2_d697a464
Finished data loading and preprocessing.
  NumNodes: 2708
  NumEdges: 10556
  NumFeats: 1433
  NumClasses: 7
  NumTrainingSamples: 140
  NumValidationSamples: 500
  NumTestSamples: 1000
Done saving data into cached files.


In [32]:
g

Graph(num_nodes=6, num_edges=4,
      ndata_schemes={'x': Scheme(shape=(3,), dtype=torch.float32)}
      edata_schemes={'x': Scheme(shape=(), dtype=torch.int32)})

In [34]:
#different names can have different shapes
g.ndata['y']=torch.randn(g.num_nodes(), 5)

In [35]:
g.ndata['x'][1] #get node 1's feature

tensor([1., 1., 1.])

In [36]:
g.edata['x'][torch.tensor([0,3])] #get features of edge 0 and 3

tensor([1, 1], dtype=torch.int32)

Example of weighted graph

In [37]:
# edges 0->1, 0->2, 0->3, 1->3
edges = torch.tensor([0, 0, 0, 1]), torch.tensor([1, 2, 3, 3])
weights = torch.tensor([0.1, 0.6, 0.9, 0.7])  # weight of each edge
g = dgl.graph(edges)
g.edata['w'] = weights  # give it a name 'w'


In [40]:
g

Graph(num_nodes=4, num_edges=4,
      ndata_schemes={}
      edata_schemes={'w': Scheme(shape=(), dtype=torch.float32)})

In [43]:
#dataset = dgl.data.KarateClubDataset()

In [59]:
d = dgl.data.SSTDataset()

KeyboardInterrupt: 

In [44]:
#dataset

In [45]:
#pwd

In [10]:
num_classes = dataset.num_classes
g = dataset[0]

print('Number of nodes:', g.num_nodes())
print('Number of edges:', g.num_edges())

Number of nodes: 34
Number of edges: 156


In [14]:
g

Graph(num_nodes=34, num_edges=156,
      ndata_schemes={'label': Scheme(shape=(), dtype=torch.int64)}
      edata_schemes={})

In [11]:
print('Node feature names:', g.ndata.keys())
print('Edge feature names:', g.edata.keys())


Node feature names: dict_keys(['label'])
Edge feature names: dict_keys([])


In [42]:
# class GCN(nn.Module):
#     def __init__(self, in_feats, n_hidden, n_classes):
#         super(GCN, self).__init__()
#         self.conv1 = GraphConv(in_feats, n_hidden)
#         self.conv2 = GraphConv(n_hidden, n_classes)
#         self.relu = nn.ReLU()
    
#     def forward(self, g, in_feat):
#         output = self.conv1(g, in_feat)
#         output = self.relu(output)
#         output = self.conv2(g, output)
#         return output 

#models = GCN(g.ndata['feat'].shape[1], 16, dataset.num_classes)

Heterogeneous Graphs

In [47]:
#createing a Heterogeneous Graph
# link for graph pic: https://www.geeksforgeeks.org/create-heterogeneous-graph-using-dgl-in-python/
# Create a heterograph with 3 node types and 3 edges types.

#(source node type, edge type, destination node type
graph_data = {
    #source   edge     destination
    ('user','watches', 'movies'): (torch.tensor([0, 0, 1, 2]), #user   #user 0 watch movie0, user 0 watch movie1, user 2 watch movie1
                                   torch.tensor([0, 1, 0, 1])),# movie
    
    ('director', 'directs', 'movie'): (torch.tensor([0, 1]),#director #director0 film movie1, director1 film movie0
                                       torch.tensor([1, 0])) #movie
                                       }

hetero_graph = dgl.heterograph(graph_data)

                                       
    


In [48]:
hetero_graph

Graph(num_nodes={'director': 2, 'movie': 2, 'movies': 2, 'user': 3},
      num_edges={('director', 'directs', 'movie'): 2, ('user', 'watches', 'movies'): 4},
      metagraph=[('director', 'movie', 'directs'), ('user', 'movies', 'watches')])

In [49]:
hetero_graph.ntypes

['director', 'movie', 'movies', 'user']

In [50]:
hetero_graph.etypes

['directs', 'watches']

In [51]:
dataset3 = dgl.data.CiteseerGraphDataset()

download failed, retrying, 4 attempts left


KeyboardInterrupt: 

In [52]:
from dgl.data.rdf import AIFBDataset, AMDataset, BGSDataset, MUTAGDataset

In [53]:
AMDataset

dgl.data.rdf.AMDataset

In [54]:
import networkx as nx


In [58]:
dataset = AIFBDataset()

download failed, retrying, 4 attempts left


KeyboardInterrupt: 

In [57]:
!pip install rdflib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 13.5 MB/s eta 0:00:0000:01
